# 🐳 Module 7.3: Docker Containerization

**Goal**: Build a Docker container from an MLFlow model and run it locally.

**Prerequisites**: Docker Desktop must be installed and running.

---

## Step 1: Verify Docker is Available

In [ ]:
import subprocess

try:
    result = subprocess.run(["docker", "--version"], capture_output=True, text=True)
    print(f"✅ {result.stdout.strip()}")
except FileNotFoundError:
    print("❌ Docker not found. Install Docker Desktop first.")

## Step 2: Train & Log a Model

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

mlflow.autolog(disable=True)
mlflow.set_experiment("07_docker_deployment")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="docker_model"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(model, "model", signature=signature, input_example=X_train[:2])
    mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test)))
    
    run_id = mlflow.active_run().info.run_id

print(f"✅ Model logged. Run ID: {run_id}")

## Step 3: Build Docker Image

MLFlow has a built-in command to create a Docker image from a logged model.

⚠️ **Run this in a terminal** (it may take a few minutes the first time):

```bash
cd c:\Users\sujat\projects\MLFlow_Learn
mlflow models build-docker -m runs:/<RUN_ID>/model -n wine-quality-model
```

Replace `<RUN_ID>` with the run ID from above.

In [ ]:
# Print the exact command to run
print("📋 Copy this command to your terminal:")
print(f"\nmlflow models build-docker -m runs:/{run_id}/model -n wine-quality-model")
print("\n⏳ This may take 2-5 minutes on first build (downloads base image).")

## Step 4: Run the Docker Container

After the image is built:

```bash
docker run -p 8080:8080 wine-quality-model
```

The model server will be available at `http://localhost:8080`.

In [ ]:
# Test the containerized model
import requests

DOCKER_URL = "http://localhost:8080"

# Prepare test data
sample = X_test[:3]
payload = {
    "dataframe_split": {
        "columns": list(sample.columns),
        "data": sample.values.tolist()
    }
}

try:
    response = requests.post(
        f"{DOCKER_URL}/invocations",
        headers={"Content-Type": "application/json"},
        json=payload
    )
    print(f"🐳 Docker container predictions: {response.json()}")
    print(f"   Actual: {list(y_test[:3])}")
    print("\n✅ Containerized model works!")
except requests.ConnectionError:
    print("❌ Container not running.")
    print("   Run: docker run -p 8080:8080 wine-quality-model")

## Step 5: Docker Management Commands

```bash
# List running containers
docker ps

# Stop a container
docker stop <container_id>

# List images
docker images | grep wine

# Remove image (cleanup)
docker rmi wine-quality-model
```

## Custom Dockerfile (Advanced)

For more control, you can also write a custom Dockerfile. See `Dockerfile` in this module directory for an example.

## 🔑 Key Takeaways

1. **`mlflow models build-docker`** creates a production-ready Docker image
2. The container exposes the same **`/invocations`** and **`/ping`** endpoints
3. Default port is **8080** inside the container
4. The image includes all dependencies — fully portable
5. This same image can be pushed to **AWS ECR** for cloud deployment

---

## ➡️ Next: `04_aws_deployment.ipynb` — Deploy to AWS